# 05. Advanced Model Experiments: Structured Roadmap

Fundamentals of Natural Language / NLP-I, Universitat Autonoma de Barcelona, academic year 2025-2026. Team 10: Phoebe Iglesias, David Redrejo, and Pau Rossell.

At this point we stop ourselves from adding advanced models randomly. We already have EDA, preprocessing decisions, classical baselines, retrieval, CLS pooling, and mean pooling. This notebook turns those results into a controlled experimental roadmap.

## 1. What We Learned So Far

**From EDA.** The dataset is short-literal ICD category prediction, not long-document ICD coding. It has 36 categories, strong class imbalance, duplicate literals, and some same-literal/different-code ambiguities. Leaderboard literals have similar length distributions, but possible distribution shift remains because we only observe text length and surface patterns.

**From preprocessing.** Biomedical Spanish text is fragile. We preserve case, accents, punctuation, digits, and abbreviations for RoBERTa. Light whitespace cleanup is the final pipeline; stronger normalization is only an ablation for classical baselines.

**From classical baselines.** Majority baseline gives 0.125 accuracy and almost zero macro F1. Character TF-IDF and word TF-IDF are strong: they reach around 0.52 accuracy and show that surface lexical patterns matter. Retrieval is intuitive but weaker than classifiers because identical or similar literals can still map to different categories.

**From CLS pooling.** RoBERTa CLS achieved the best validation accuracy so far: 0.5693. This suggests that the Spanish biomedical-clinical pretrained language model adds value beyond sparse TF-IDF features.

**From mean pooling.** Mean pooling did not beat CLS in accuracy, but it slightly improved macro F1. This suggests that pooling choices affect class balance and not only the overall score.

In [ ]:
import pandas as pd

model_summary = pd.DataFrame([
    {'model': 'v00_majority_baseline', 'accuracy': 0.125182, 'macro_f1': 0.006181, 'weighted_f1': 0.027854},
    {'model': 'v01_tfidf_char_logreg', 'accuracy': 0.522628, 'macro_f1': 0.402554, 'weighted_f1': 0.494943},
    {'model': 'v02_tfidf_word_svm', 'accuracy': 0.520073, 'macro_f1': 0.474196, 'weighted_f1': 0.514018},
    {'model': 'v03_similarity_retrieval_baseline', 'accuracy': 0.497445, 'macro_f1': 0.462789, 'weighted_f1': 0.496120},
    {'model': 'v04_roberta_cls', 'accuracy': 0.569343, 'macro_f1': 0.494329, 'weighted_f1': 0.554347},
    {'model': 'v05_roberta_mean', 'accuracy': 0.564599, 'macro_f1': 0.496567, 'weighted_f1': 0.549541},
])
model_summary


## 2. Weaknesses That Remain

- **Class imbalance:** some categories remain rare, and rare labels such as `A`, `W`, and `X` have zero recall in both CLS and mean pooling validation reports.
- **Ambiguous short literals:** nearest-neighbor analysis showed cases where exact or near-exact literals point to different categories. Short text lacks the clinical context that would resolve ambiguity.
- **Categories with low recall:** RoBERTa improves accuracy but does not solve minority categories. Macro F1 remains close to classical baselines.
- **Similar categories confused:** broad ICD categories can share surface terminology, especially when procedures, diagnoses, and history codes are written compactly.
- **Possible leaderboard distribution shift:** train and leaderboard look similar in length, but we do not observe leaderboard labels, so lexical or category distribution shift is still possible.
- **Overfitting/underfitting:** training loss keeps decreasing after the best validation epoch while validation loss rises, so later experiments should manage regularization and schedules.

In [ ]:
cls_per_class = pd.read_csv('../outputs/metrics/v04_roberta_cls_per_class_metrics.csv')
mean_per_class = pd.read_csv('../outputs/metrics/v05_roberta_mean_per_class_metrics.csv')
cls_per_class.sort_values('recall').head(10)


## 3. Candidate Improvements

The table below is the roadmap. Each candidate is tied to evidence from EDA, the course material, the ICD coding survey, or our observed model behavior. This is why we did not add advanced models randomly: every next experiment should answer a weakness we had already observed. The goal is not to implement everything; the goal is to choose experiments that answer a specific question.

In [ ]:
roadmap = pd.read_csv('../reports/tables/advanced_experiment_roadmap.csv')
roadmap


## 4. Decisions

**Implement next.** We prioritize class-weighted loss, learning-rate tuning, warmup scheduler, dropout tuning, ensembling, and calibration/confidence analysis. These are connected to observed weaknesses and have low-to-medium implementation cost.

**Maybe.** Focal loss, max-length tuning, freezing/unfreezing, label smoothing, and safe data augmentation are plausible but should be tested only after the first priority experiments.

**Future work.** Layer-wise learning-rate decay and pseudo-labeling are interesting but more complex or riskier. They are better framed as future work unless we have time and a stable validation protocol.

## 5. Why This Roadmap Fits the Course Story

This roadmap follows the project story from the course: corpora and annotation analysis first, then text processing and sparse vector-space baselines, then pretrained Transformer models, and finally controlled improvement experiments.

The survey helped us understand why ICD coding is hard: imbalance, terminology, hierarchy, and interpretability. Our Kaggle task is simpler than full multi-label ICD coding, but the same ideas still guide our decisions. The next experiments should therefore be justified by the data and by the course concepts, not by trial-and-error model stacking.

## 6. First Roadmap Implementation: Imbalance-Aware Losses

The first advanced experiment we implemented was imbalance-aware loss design. This directly follows the EDA long-tail category distribution and the ICD survey discussion of unbalanced ICD labels. The goal was not only to improve accuracy, but also to test whether rare-category recall and macro F1 could improve.

## v06 Setup

`models/v06_roberta_mean_imbalance_aware.py` starts from the mean-pooling RoBERTa baseline. Class weights are computed from the training split only, so validation and leaderboard labels are not leaked into the loss. We tested class-weighted CrossEntropyLoss and focal loss with gamma values 1 and 2. The standard v05 mean-pooling model is included as the reference row.

In [ ]:
imbalance_grid = pd.read_csv('../reports/tables/v06_imbalance_aware_grid.csv')
imbalance_grid


## v06 Result and Decision

| model / loss | validation accuracy | macro F1 | weighted F1 | decision |
|---|---:|---:|---:|---|
| v05 standard mean CE | 0.5646 | 0.4966 | 0.5495 | reference remains stronger by accuracy |
| v06 focal gamma 1 | 0.5573 | 0.4804 | 0.5398 | best v06 by accuracy, not final |
| v06 focal gamma 2 | 0.5555 | 0.4928 | 0.5424 | close but not better than v05 |
| v06 class-weighted CE | 0.5445 | 0.5183 | 0.5375 | best macro F1, too much accuracy loss |

The result is a useful trade-off. Class-weighted CE improves macro F1, which means it addresses the imbalance problem better than standard mean pooling. However, it loses too much accuracy for the competition objective. Focal loss preserves accuracy better than class weighting, but does not beat the standard mean-pooling baseline.

Therefore, `v06` should be kept as an imbalance ablation, not the final model. CLS remains the current candidate final model by validation accuracy.

In [ ]:
recall_vs_v05 = pd.read_csv('../reports/tables/v06_per_class_recall_vs_v05.csv')
recall_vs_v05.sort_values('recall_delta_v06_minus_v05', ascending=False).head(10)


## What This Taught Us

The class-weighted model improves recall for some categories, but it also damages others. This confirms that imbalance is not solved by a single loss change. The result was useful even when it was not the final model because it showed the trade-off between accuracy and rare-class behavior. The next steps should combine imbalance awareness with error analysis and possibly ensembling, rather than simply replacing the baseline loss.

For the report, this is a good example of students learning from an experiment that does not become the final model: it answers a question, reveals a trade-off, and gives evidence for why the project moves toward ensembling/calibration instead of blindly applying more aggressive loss functions.

## v09 Ensemble After Individual Models

After reading the survey and after running several individual models, we reached the point where ensembling made sense. We did not start with an ensemble because that would have hidden the learning process. First we needed EDA, preprocessing, classical baselines, RoBERTa CLS, RoBERTa mean pooling, imbalance-aware variants, and safe data-strategy experiments.

From Fundamentals of Machine Learning, the idea is that an ensemble can reduce variance when the models make partly different errors. In our case, TF-IDF character n-grams and RoBERTa are different representations: TF-IDF is strong on surface forms, abbreviations, punctuation, and character fragments, while RoBERTa uses contextual biomedical Spanish subword representations. However, ensembling can fail if all models make correlated errors on the same ambiguous literals.


In [ ]:
import pandas as pd
ensemble = pd.read_csv('../reports/tables/v09_ensemble_comparison.csv')
ensemble[['candidate_version', 'kind', 'accuracy', 'macro_f1', 'weighted_f1', 'recipe']].head(12)


### Ensemble Recipes Tested

We tested probability averaging for neural models, weighted probability averaging, majority voting over predicted labels, and a fallback rule where TF-IDF can override the neural prediction only when neural confidence is low and TF-IDF confidence is high. The fallback is intentionally conservative because TF-IDF probabilities are not guaranteed to be calibrated like neural probabilities.

We did **not** use leaderboard labels. We also did not tune recipes against public leaderboard feedback. The selected recipe is based on internal validation accuracy.


### Result and Decision

The best ensemble was majority vote over `v04_roberta_cls`, `v05_roberta_mean`, `v08_roberta_mean_dedupe`, `v08_roberta_mean_weighted_sampler`, and `v01_tfidf_char_logreg`, with average-probability tie-breaking.

Validation result:

| model | accuracy | macro F1 | weighted F1 |
|---|---:|---:|---:|
| best single model, `v04_roberta_cls` | 0.5693 | 0.4943 | 0.5543 |
| selected `v09_majority_vote` ensemble | 0.5766 | 0.5063 | 0.5615 |

This is the first result that clearly improves over the CLS baseline on accuracy and also improves macro F1. It becomes the current final candidate, with the caveat that the final report must describe it as a validation-driven ensemble, not as a public-leaderboard-tuned system.


### Future Work

A natural future direction is to compare our model family with other podium teams' systems after the competition, and to test whether a podium-style ensemble is allowed and academically appropriate. That would only be reported as future work unless the rules and supervision explicitly allow it.
